# VLADE · S2 CoT + Action KD · 2026-11-25

GitHub의 [CoT KD 실행 코드](https://github.com/eastha10/VLADE/blob/4b24ece9dbdc5cb690a1bab189c121af85d95967/experiments/colab_teacher_plan_kd.ipynb)와
기존 `VLADE_Action_KD_Ready_20261125.ipynb`의 설정·로그·저장 검사·재개·결과 묶음을 합쳤습니다.
소스는 실행 시 main의 불변 commit으로 확정하고
`src/vlade/legacy/plan_cot/configs/experiments/qwen_vla_plan_kd.json`과 공식 `vlade.research.distillation.job`을 호출합니다.

**입력:** 기존 고정 AVA T1을 그대로 사용합니다. T1을 추가 학습하거나 Plan 정답 주석을 준비하지 않습니다.
T1이 현재 관측·지시에서 직접 생성한 Plan을 S2 입력에 넣습니다. Plan 학습 이력이 없다는 이유로 생성을 막지 않습니다.
빈 출력·EOS 누락·잘못된 형식은 실제 생성 결과에서 검사하며 Plan의 의미와 액션 일치는 probe 원문으로 확인합니다.
현재 A100 세션은 별도 existing-session probe 노트북의 코드 셀을 복사해 재설치·재시작 없이 확인할 수 있습니다.

1. Colab 새 **BF16 GPU 런타임**과 Secrets의 `HF_TOKEN` 읽기·쓰기 권한을 준비합니다. A100급에서 파일럿으로 메모리를 확인합니다.
2. 설정 → 소스 → 계획/재개 메모 → setup → preflight → **T1 플랜 probe**를 실행합니다.
3. probe 원문에서 관측·지시·행동과 플랜의 일치 여부를 확인합니다. 형식 통과는 의미 품질의 증명이 아닙니다.
4. `RUN_PILOT=True`로 1 optimizer-step 파일럿과 저장 검사를 실행합니다.
5. 통과 후 `RUN_TRAIN=True`로 같은 run의 목표 step까지 이어갑니다.

Student는 **T1 플랜을 입력에 추가**하고 `GT L1 + λ × Teacher Action L1`으로 학습합니다.
Student 언어 CE 학습은 아닙니다. 기본값은 전체 Student 학습, 누적 8회, LR 2e-5, 100 step, 25 step마다 저장, 검증 query 32개, seed 42입니다.
S2는 비공개 `VLADE-konkuk/VLADE-checkpoints`의 `runs/<run-group>/S2`에 저장합니다.

환경 설치·다운로드·GPU 실행·HF 게시는 해당 셀을 실행할 때 발생합니다. 이 파일을 만든 시점에는 실제 학습을 실행하지 않았습니다.
[Plan 조건](https://github.com/eastha10/VLADE/blob/4b24ece9dbdc5cb690a1bab189c121af85d95967/docs/environment/teacher_plan_generation.md) ·
[저장·재개](https://github.com/eastha10/VLADE/blob/4b24ece9dbdc5cb690a1bab189c121af85d95967/docs/environment/hugging_face_checkpoints.md)


In [ ]:
from pathlib import Path
import os, sys, re, json, time, subprocess, uuid, copy, math
from datetime import datetime, timezone

SOURCE_COMMIT = ''  # 새 실행: 소스 셀에서 main의 SHA 확정. 재개: 최초 실행의 SHA 지정
WORKSPACE = Path('/content/vlade_distillation')  # 이전 검증과 같은 HF 캐시 재사용
CONFIG = 'src/vlade/legacy/plan_cot/configs/experiments/qwen_vla_plan_kd.json'
MAX_STEPS = 100
SAVE_STEPS = 25
ACCUMULATION_STEPS = 8
VALIDATION_QUERIES = 32
KD_WEIGHT = 1.0
PLAN_MAX_NEW_TOKENS = 48  # EOS 포함 생성 예산: 1~128
# 기본은 기존 고정 T1입니다. 다른 HF run checkpoint를 선택할 때만 pin을 지정합니다.
TEACHER_ASSET_KEY = 'T1_plan'  # 기존 T1 asset을 덮어쓰지 않는 별도 입력 키
TEACHER_CHECKPOINT_PIN = {}  # {'repo_id':'VLADE-konkuk/VLADE-checkpoints', 'revision':'40자리 HF SHA', 'prefix':'runs/<group>/<output>/checkpoints/step-0000100'}
# 새 실행은 비워 둡니다. 재개는 보관한 기존 run group과 불변 HF revision을 지정합니다.
RESUME_RUN_GROUP = ''
RESUME_REVISIONS = {}  # 예: {'S2': '<40자리 HF commit SHA>'}
RUN_PILOT = False  #@param {type:'boolean'}
RUN_TRAIN = False  #@param {type:'boolean'}
RETRY_UPLOAD = False  #@param {type:'boolean'}

assert not SOURCE_COMMIT or re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT)
assert type(MAX_STEPS) is int and MAX_STEPS > 1
assert type(SAVE_STEPS) is int and 1 <= SAVE_STEPS <= MAX_STEPS
assert type(ACCUMULATION_STEPS) is int and ACCUMULATION_STEPS >= 1
assert type(VALIDATION_QUERIES) is int and VALIDATION_QUERIES >= 1
assert type(KD_WEIGHT) in (int,float) and math.isfinite(KD_WEIGHT) and KD_WEIGHT > 0
assert type(PLAN_MAX_NEW_TOKENS) is int and 1 <= PLAN_MAX_NEW_TOKENS <= 128
assert isinstance(TEACHER_CHECKPOINT_PIN, dict)
if TEACHER_CHECKPOINT_PIN:
    assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', TEACHER_ASSET_KEY) and '..' not in TEACHER_ASSET_KEY
    assert TEACHER_ASSET_KEY not in {'T1', 'S0', 'prepared_d_seed42'}
    assert set(TEACHER_CHECKPOINT_PIN) == {'repo_id', 'revision', 'prefix'}
    assert TEACHER_CHECKPOINT_PIN['repo_id'] == 'VLADE-konkuk/VLADE-checkpoints'
    assert re.fullmatch(r'[0-9a-f]{40}', TEACHER_CHECKPOINT_PIN['revision'])
    assert re.fullmatch(r'runs/[^/]+/[^/]+/checkpoints/step-[0-9]{7,}', TEACHER_CHECKPOINT_PIN['prefix'])
assert isinstance(RESUME_REVISIONS, dict) and set(RESUME_REVISIONS) <= {'S2'}
assert not RESUME_REVISIONS or RESUME_RUN_GROUP, 'HF 복구 시 기존 RESUME_RUN_GROUP도 필요합니다.'
assert all(re.fullmatch(r'[0-9a-f]{40}', v) for v in RESUME_REVISIONS.values())
if RESUME_RUN_GROUP:
    assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,79}', RESUME_RUN_GROUP)
    if globals().get('RUN_GROUP') and RUN_GROUP != RESUME_RUN_GROUP:
        raise RuntimeError('다른 run이 이미 선택되어 있습니다. 새 런타임에서 재개하세요.')
    RUN_GROUP = RESUME_RUN_GROUP
else:
    RUN_GROUP = globals().get('RUN_GROUP') or ('qwen-cot-action-kd-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8])
OVERRIDES = {'max_steps': MAX_STEPS, 'save_steps': SAVE_STEPS,
             'accumulation_steps': ACCUMULATION_STEPS, 'validation_queries': VALIDATION_QUERIES,
             'distillation.action_kd_weight': KD_WEIGHT,
             'distillation.teacher_plan_max_new_tokens': PLAN_MAX_NEW_TOKENS}
if TEACHER_CHECKPOINT_PIN:
    OVERRIDES['workflow.checkpoint_pins'] = {TEACHER_ASSET_KEY: copy.deepcopy(TEACHER_CHECKPOINT_PIN)}
    OVERRIDES['distillation.teacher_checkpoint'] = 'hfasset://' + TEACHER_ASSET_KEY
TEACHER_INPUT_KEY = TEACHER_ASSET_KEY if TEACHER_CHECKPOINT_PIN else 'T1'
RUN_ROOT = WORKSPACE / 'runs' / RUN_GROUP
OUTPUT = RUN_ROOT / 'S2'
print('RUN_GROUP:', RUN_GROUP, '| 고정 소스:', SOURCE_COMMIT)
print('HF:', 'VLADE-konkuk/VLADE-checkpoints', '| 경로:', 'runs/' + RUN_GROUP + '/S2')
print('동일 run group/output에 여러 프로세스를 실행하지 마세요.')


## 1. 검증한 소스 고정

기존 실행 소스를 변경하지 않고 동일 commit을 재사용합니다. 다른 commit의 모듈이 이미 로드되어 있으면 중단합니다.
초기 소스 clone만 수행하며 Git branch·submodule revision을 변경하지 않습니다.


In [ ]:
REPOSITORY = 'https://github.com/eastha10/VLADE.git'
if not SOURCE_COMMIT:
    SOURCE_COMMIT = subprocess.check_output(['git','ls-remote',REPOSITORY,'refs/heads/main'], text=True).split()[0]
assert re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT)
CODE = Path('/content/vlade_sources') / SOURCE_COMMIT
os.environ.setdefault('VLADE_SESSION_STARTED_UNIX', str(time.time()))
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
for name, module in list(sys.modules.items()):
    location = getattr(module, '__file__', None)
    if (name == 'vlade' or name.startswith('vlade.')) and location and not Path(location).resolve().is_relative_to(CODE):
        raise RuntimeError('다른 VLADE 소스가 로드되어 있습니다. 런타임을 재시작하세요.')
if not CODE.exists():
    CODE.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git','clone','--no-checkout','--filter=blob:none',REPOSITORY,str(CODE)],check=True)
    subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
    subprocess.run(['git','-C',str(CODE),'checkout','--detach',SOURCE_COMMIT],check=True)
sys.path[:0] = [str(CODE/'src'), str(CODE)]
from vlade.utils.source_identity import source_identity
assert source_identity(CODE, require_clean=True)['git_commit'] == SOURCE_COMMIT
from vlade.legacy.plan_cot.distillation.teacher_plan import plan_generation_provenance
from vlade.research.distillation import job, load_settings
workflow, variants = load_settings(CODE / CONFIG, OVERRIDES)
assert list(variants) == ['gt_kd']
effective = variants['gt_kd']
assert effective['initial_checkpoint'] == 'hfasset://S0'
assert effective['data'] == 'hfasset://prepared_d_seed42'
assert effective['distillation']['teacher_checkpoint'] == 'hfasset://' + TEACHER_INPUT_KEY
assert effective['distillation']['teacher_mode'] == 'inference'
assert effective['distillation']['teacher_plan_max_new_tokens'] == PLAN_MAX_NEW_TOKENS
assert effective['distillation']['task_weight'] == 1.0
assert workflow['registry_revision'] == '195d32e54c7f48960411c32963d91746183de488'
_CONTRACT = json.dumps({'commit':SOURCE_COMMIT, 'group':RUN_GROUP, 'workspace':str(WORKSPACE),
                        'config':CONFIG, 'overrides':OVERRIDES}, sort_keys=True)
print(json.dumps({'workflow':workflow, 'effective_settings':effective}, ensure_ascii=False, indent=2))


## 2. 실행 계획과 재개 메모

이 셀은 모델을 로드하지 않으며 GPU 학습이나 HF 업로드를 시작하지 않습니다.
다운로드한 메모 JSON을 보관하세요. 재개할 때는 같은 commit·run group·설정·workspace를 사용합니다.


In [ ]:
import contextlib
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, text):
        for stream in self.streams: stream.write(text)
        return len(text)
    def flush(self):
        for stream in self.streams: stream.flush()

def run_action(action):
    current = json.dumps({'commit':SOURCE_COMMIT, 'group':RUN_GROUP, 'workspace':str(WORKSPACE),
                          'config':CONFIG, 'overrides':OVERRIDES}, sort_keys=True)
    if current != _CONTRACT:
        raise RuntimeError('설정이 바뀌었습니다. 새 실행은 새 run group으로 준비하세요.')
    if action == 'preflight':
        globals()['PREFLIGHT_OK'] = False
        globals()['PROBE_OK'] = False
    elif action == 'probe':
        globals()['PROBE_OK'] = False
    logs = RUN_ROOT / 'launch-logs'
    logs.mkdir(parents=True, exist_ok=True)
    path = logs / (action + '-' + uuid.uuid4().hex[:8] + '.log')
    # 공식 실행기가 자식 프로세스에만 Colab Secret을 전달합니다.
    with path.open('x', encoding='utf-8') as stream, contextlib.redirect_stdout(Tee(sys.stdout, stream)):
        job(CODE / CONFIG, workspace=WORKSPACE, run_group=RUN_GROUP, action=action,
            overrides=copy.deepcopy(OVERRIDES), resume_revisions=copy.deepcopy(RESUME_REVISIONS), colab=True)
    print('실행 로그:', path)

run_action('plan')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
memo = {'source_commit':SOURCE_COMMIT, 'run_group':RUN_GROUP, 'workspace':str(WORKSPACE),
        'config':CONFIG, 'overrides':OVERRIDES,
        'registry_revision':workflow['registry_revision'], 'hf_repo':'VLADE-konkuk/VLADE-checkpoints',
        'hf_run':RUN_GROUP+'/S2',
        'resume_note':'동일 설정과 RESUME_RUN_GROUP 사용. RESUME_REVISIONS에는 보관한 불변 HF commit SHA 지정.'}
MEMO = RUN_ROOT / 'resume-settings.json'
if MEMO.exists() and json.loads(MEMO.read_text()) != memo:
    raise RuntimeError('기존 재개 메모와 설정이 다릅니다.')
MEMO.write_text(json.dumps(memo, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(memo, ensure_ascii=False, indent=2))
from google.colab import files
files.download(str(MEMO))


## 3. 공식 Student·AVA 환경 준비

이 셀은 패키지를 설치하지만 학습은 시작하지 않습니다. 두 환경의 Transformers 버전은 분리됩니다.
기존 `WORKSPACE/environments`가 검증된 상태면 재사용합니다. Colab 호스트 Python에 AVA fork를 설치하지 않습니다.


In [ ]:
run_action('setup')
print('환경 준비 완료. 다음 preflight에서 실제 HF 입력·봉인·S0/T1 데이터 계약·비공개 저장소를 확인합니다.')


## 4. 학습 전 검사

고정 HF 입력의 무결성·S0/T1 데이터 계약·비공개 Hub 연결을 확인합니다.
기존 Action-only T1의 직접 Plan 생성을 허용합니다. Plan 감독 이력이 있으면 그 기록도 검증합니다.
BF16 GPU 점검까지 완료해야 `PREFLIGHT_OK=True`가 됩니다.
실제 생성·학습·원격 checkpoint 쓰기는 다음 단계에서 각각 확인합니다.


In [ ]:
run_action('preflight')
inputs = json.loads((RUN_ROOT/'inputs.json').read_text(encoding='utf-8'))
print('실제 입력 pin:')
for key in ('S0',TEACHER_INPUT_KEY,'prepared_d_seed42'):
    record = inputs['assets'][key]
    assert re.fullmatch(r'[0-9a-f]{40}', record['revision'])
    print(key, record['repo_id'], record['revision'])
from vlade.runtime.environment import locate, runtime_environment
STUDENT_PYTHON = locate(CODE, WORKSPACE, 'student')
check_env = runtime_environment(WORKSPACE)
check_env['PYTHONPATH'] = str(CODE/'src')
subprocess.run([str(STUDENT_PYTHON), '-B', '-c',
    "import torch,json; assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'BF16 GPU required'; print(json.dumps({'gpu':torch.cuda.get_device_name(0),'free_GiB':torch.cuda.mem_get_info()[0]/1024**3,'torch':torch.__version__}))"],
    cwd=CODE, env=check_env, check=True)
PREFLIGHT_OK = True
print('입력·Hub·BF16 확인 완료. 다음 probe에서 실제 T1 플랜을 확인하세요.')


## 5. T1 플랜 생성 probe

Student를 로드하지 않고 Train·Validation 첫 Episode에서 각각 최대 2 query를 확인합니다.
Train은 액션과 플랜, Validation은 플랜만 요청합니다. Validation의 `action_shape=null`은 요청하지 않은 항목입니다.
빈 플랜·생성 실패·미완료 보고서는 통과시키지 않습니다. 원문을 보고 의미 품질을 별도로 확인하십시오.


In [ ]:
assert globals().get('PREFLIGHT_OK'), 'preflight 셀을 먼저 통과하세요.'
PROBE_OK = False
prior_probes = set(RUN_ROOT.glob('teacher-plan-probe-*.jsonl'))
run_action('probe')
new_probes = set(RUN_ROOT.glob('teacher-plan-probe-*.jsonl')) - prior_probes
if len(new_probes) != 1:
    raise RuntimeError('이번 호출의 probe 보고서를 하나만 확인할 수 있어야 합니다.')
PROBE_PATH = next(iter(new_probes))
probe_rows = [json.loads(line) for line in PROBE_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
if not probe_rows or probe_rows[-1].get('status') != 'complete':
    raise RuntimeError('T1 probe 미완료입니다. 보고서 오류를 확인하세요.')
if any(row.get('status') in {'failed','plan_error'} for row in probe_rows):
    raise RuntimeError('실패한 플랜이 포함되어 있습니다.')
queries = [row for row in probe_rows if row.get('status') == 'ok']
if not queries or probe_rows[-1].get('queries') != len(queries):
    raise RuntimeError('probe query 수가 일치하지 않습니다.')
if {row['split'] for row in queries} != {'train','validation'}:
    raise RuntimeError('Train·Validation 양쪽 probe가 필요합니다.')
for row in queries:
    if not isinstance(row.get('plan'), str) or not row['plan'].strip():
        raise RuntimeError('빈 플랜은 허용하지 않습니다.')
    if row['split'] == 'train' and row.get('action_shape') != [8,7]:
        raise RuntimeError('Teacher 액션 형태가 [8,7]이 아닙니다.')
    print(json.dumps({key:row.get(key) for key in ('split','episode_id','timestep','instruction','plan','action_shape','action_status')}, ensure_ascii=False, indent=2))
PROBE_OK = True
print('probe 형식·액션 계약 확인:', PROBE_PATH)
print('플랜의 관측 반영·행동 일치·성공률은 별도 평가가 필요합니다.')


## 6. 파일럿·학습 결과 검사 도구

로컬 봉인과 CoT KD 손실·실제 Teacher identity를 검사합니다. 원격 검사는 조회 시점의 HF commit을 고정한 후
`last.json`·`COMPLETE`·checkpoint manifest를 로컬과 대조합니다. 전체 가중치 재다운로드 검사는 아닙니다.
원격 검증 실패 시 성공으로 표시하지 않고 pending 여부를 확인합니다. 모델/원본 데이터는 결과 ZIP에 포함하지 않습니다.


In [ ]:
AUDIT_SOURCE = r"""
import json, math, os, sys
from pathlib import Path
from vlade.training.run_control import verify_checkpoint
from vlade.legacy.plan_cot.distillation.teacher_plan import plan_generation_provenance
from vlade.storage.assets import sha256
from huggingface_hub import HfApi, hf_hub_download
output, destination = map(Path, sys.argv[1:3])
group, target, mode = sys.argv[3], int(sys.argv[4]), sys.argv[5]
report = {'status':'FAIL', 'mode':mode}
try:
    checkpoints = sorted(output.glob('step-*'))
    if not checkpoints: raise ValueError('No checkpoint')
    checkpoint = checkpoints[-1]
    identity = json.loads((checkpoint/'identity.json').read_text())
    settings = identity['settings']
    seal = verify_checkpoint(checkpoint, accumulation=settings['accumulation_steps'])
    assert settings['distillation']['teacher_plan_max_new_tokens'] > 0
    assert settings['distillation']['teacher_mode'] == 'inference'
    assert identity['teacher_inference']['frozen'] is True
    teacher = identity['teacher_inference']
    plan_generation_provenance({'step': teacher['step'], 'plan_supervision': teacher.get('plan_supervision')})
    assert seal['step'] >= 1 and seal['microsteps'] == seal['step'] * settings['accumulation_steps']
    metrics = json.loads((checkpoint/'metrics.json').read_text())
    for key in ('loss','task_l1','action_kd_l1','validation_l1','grad_norm'):
        assert type(metrics[key]) in (float,int) and math.isfinite(metrics[key]) and metrics[key] >= 0, key
    if mode == 'pilot': assert metrics['grad_norm'] > 0, 'No nonzero gradient observed in latest checkpoint'
    expected = settings['distillation']['task_weight'] * metrics['task_l1'] + settings['distillation']['action_kd_weight'] * metrics['action_kd_l1']
    assert math.isclose(metrics['loss'], expected, rel_tol=1e-4, abs_tol=1e-6), 'KD loss terms mismatch'
    assert (checkpoint/'trainer_state.pt').is_file()
    pending = output/'hub_pending.json'
    assert not pending.exists() or not json.loads(pending.read_text())['queue'], 'Hub upload pending: use retry'
    api = HfApi()
    info = api.model_info('VLADE-konkuk/VLADE-checkpoints')
    assert info.private, 'Expected private repository'
    revision = info.sha
    base = 'runs/'+group+'/S2'
    def read(relative):
        return Path(hf_hub_download('VLADE-konkuk/VLADE-checkpoints', relative, revision=revision)).read_bytes()
    pointer = json.loads(read(base+'/last.json'))
    prefix = base+'/checkpoints/'+checkpoint.name
    assert pointer['checkpoint'] == prefix and pointer['step'] == seal['step']
    assert read(prefix+'/COMPLETE').strip() == b'ok'
    assert read(prefix+'/checkpoint_manifest.json') == (checkpoint/'checkpoint_manifest.json').read_bytes()
    pin = {'S2':{'repo_id':'VLADE-konkuk/VLADE-checkpoints','revision':revision,'prefix':prefix}}
    if mode == 'final':
        assert seal['step'] == target, 'Target step not reached'
        final = json.loads(read(base+'/final.json'))
        assert final['checkpoint'] == prefix and final['step'] == target
    report.update(status='PASS_FINAL_METADATA' if mode=='final' else 'PASS_PILOT_METADATA',
                  step=seal['step'], microsteps=seal['microsteps'],
                  metrics=metrics, initial_student_step=identity['initial_student_step'],
                  teacher=identity['teacher_inference'], CHECKPOINT_PINS=pin,
                  checkpoint_manifest_sha256=sha256(checkpoint/'checkpoint_manifest.json'),
                  remote_verification='pinned pointer/COMPLETE/manifest match; weight bytes not redownloaded',
                  pending=False, rollout='NOT_RUN')
except Exception as exc:
    report.update(error_type=type(exc).__name__, error=str(exc))
    raise
finally:
    destination.write_text(json.dumps(report,ensure_ascii=False,indent=2), encoding='utf-8')
"""

def audit_checkpoint(mode):
    from google.colab import userdata
    from vlade.runtime.environment import locate, runtime_environment
    python = locate(CODE, WORKSPACE, 'student')
    reports = RUN_ROOT / 'reports'
    reports.mkdir(exist_ok=True)
    report = reports/(mode+'-audit-'+uuid.uuid4().hex[:8]+'.json')
    env = runtime_environment(WORKSPACE)
    env.update(PYTHONPATH=str(CODE/'src'), HF_TOKEN=userdata.get('HF_TOKEN'))
    if not env['HF_TOKEN']: raise RuntimeError('HF_TOKEN 접근 권한이 필요합니다.')
    result = subprocess.run([str(python),'-B','-c',AUDIT_SOURCE,str(OUTPUT),str(report),RUN_GROUP,str(MAX_STEPS),mode],
                            cwd=CODE, env=env)
    if report.exists():
        value = json.loads(report.read_text(encoding='utf-8'))
        print(json.dumps(value,ensure_ascii=False,indent=2))
        if not result.returncode: return value
    raise RuntimeError('Checkpoint/Hub 검사 실패. 위 오류와 hub_pending.json을 확인하세요.')


## 7. 실제 1-step 파일럿

`RUN_PILOT=True`를 적용하고 preflight와 probe를 통과한 뒤 실행합니다.
8 microstep에서 T1 액션·플랜 입력, Student gradient/optimizer, held-out GT L1, 저장·HF 업로드를 확인합니다.
전체 목표·schedule·손실 설정은 유지하며 1 optimizer step에서 멈춥니다.
이미 checkpoint가 있으면 공식 실행기는 재학습 대신 계약을 검사합니다.


In [ ]:
if RUN_PILOT:
    assert globals().get('PREFLIGHT_OK'), 'preflight 셀을 먼저 통과하세요.'
    assert globals().get('PROBE_OK'), 'T1 플랜 probe를 먼저 통과하세요.'
    run_action('pilot')
    PILOT_REPORT = audit_checkpoint('pilot')
    print('파일럿 저장·CoT KD 손실·HF metadata 확인 완료. 성능 향상/rollout은 미검증입니다.')
else:
    print('파일럿 미실행. 준비가 되면 첫 설정 셀의 RUN_PILOT=True를 적용하세요.')


## 8. 같은 실행에서 목표 step까지 이어 학습

파일럿·원격 저장 확인 후 `RUN_TRAIN=True`로 실행합니다. 최초 pilot 없이 train으로 시작하면 공식 실행기가 거절합니다.
기존 checkpoint의 optimizer·scheduler·RNG·query cursor·teacher 상태 재생 계약을 유지합니다.
완료 후 `CHECKPOINT_PINS`를 보관하세요. `final`은 목표 step 완료이며 rollout 성공률이나 KD 성능 향상을 의미하지 않습니다.


In [ ]:
if RUN_TRAIN:
    assert globals().get('PREFLIGHT_OK'), 'preflight 셀을 먼저 통과하세요.'
    assert globals().get('PROBE_OK'), 'T1 플랜 probe를 먼저 통과하세요.'
    if not list(OUTPUT.glob('step-*')):
        raise RuntimeError('파일럿 checkpoint가 없습니다. 신규 실행은 먼저 pilot을 실행하세요.')
    # 새 런타임에서 복구했더라도 학습 전 현재 checkpoint와 HF 게시 상태를 다시 확인합니다.
    audit_checkpoint('pilot')
    run_action('train')
    FINAL_REPORT = audit_checkpoint('final')
    CHECKPOINT_PINS = FINAL_REPORT['CHECKPOINT_PINS']
    print('평가에 사용할 고정 pin:', json.dumps(CHECKPOINT_PINS,indent=2))
else:
    print('목표 step 학습 미실행. 파일럿 통과 후 RUN_TRAIN=True를 적용하세요.')


## 9. 업로드 실패 재시도와 재개

`hub_pending.json`에 항목이 있으면 원본 checkpoint를 보존한 채 `RETRY_UPLOAD=True`로 재시도합니다.
새 Colab 세션의 복구는 처음 설정에 원래 `RESUME_RUN_GROUP`과 `RESUME_REVISIONS={'S2':'<저장해 둔 HF commit SHA>'}`를 지정합니다.
원래 source commit·설정·workspace를 유지한 채 setup → preflight → probe를 실행하면 `last`에서 봉인 검증 후 복구합니다.
이미 목표 step에 도달했다면 train은 재학습하지 않습니다. 현재 실행 출력을 다른 데이터·S0·T1으로 재사용하지 마세요.


In [ ]:
if RETRY_UPLOAD:
    run_action('retry')
    RETRY_REPORT = audit_checkpoint('final' if (OUTPUT / ('step-'+str(MAX_STEPS).zfill(7))).exists() else 'pilot')
else:
    print('업로드 retry 미실행. 정상 완료 상태에서는 필요하지 않습니다.')


## 10. 작은 결과 묶음 다운로드

재개 메모·적용 설정·입력 revision 요약·실행 로그·검사 보고·학습 지표·플랜 probe/학습 로그·checkpoint manifest만 묶습니다.
원본 데이터·모델·optimizer·토큰·전체 inputs registry는 포함하지 않습니다.
이 ZIP은 가중치 백업이 아닙니다. HF 성공 revision과 pending 부재를 확인한 다음 런타임을 종료하세요.


In [ ]:
import zipfile
bundle = RUN_ROOT / ('CoT_Action_KD_Reports_' + RUN_GROUP + '.zip')
selected = [MEMO, RUN_ROOT/'run-contract.json', RUN_ROOT/'gt_kd.json']
for folder, pattern in [(RUN_ROOT/'launch-logs','*.log'), (RUN_ROOT/'reports','*.json'),
                        (OUTPUT,'metrics-*.jsonl'), (OUTPUT,'timings-*.csv'), (OUTPUT,'teacher-plans-*.jsonl'),
                        (RUN_ROOT,'teacher-plan-probe-*.jsonl')]:
    selected.extend(sorted(folder.glob(pattern)))
selected.extend(sorted(OUTPUT.glob('step-*/metrics.json')))
selected.extend(sorted(OUTPUT.glob('step-*/checkpoint_manifest.json')))
pin_summary = RUN_ROOT/'input-pins-summary.json'
if (RUN_ROOT/'inputs.json').exists():
    registry = json.loads((RUN_ROOT/'inputs.json').read_text())
    pin_summary.write_text(json.dumps({key:{k:registry['assets'][key].get(k) for k in ('repo_id','revision','prefix')}
                                       for key in ('S0',TEACHER_INPUT_KEY,'prepared_d_seed42')},indent=2))
    selected.append(pin_summary)
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(set(selected)):
        if file.is_file(): archive.write(file,arcname=file.relative_to(RUN_ROOT))
print('학습 결과와 가중치 게시 여부는 reports의 실제 판정을 확인하세요.')
print('보고서 ZIP:', bundle, '| bytes:', bundle.stat().st_size)
from google.colab import files
files.download(str(bundle))
